# NB 06 — Evaluation & Metriken

Lädt die Raw-Result-CSVs aller drei Systeme und erstellt:
- Gruppe A: Hit Rate vs. Skalierung (Linienplot)
- Gruppe A: Skalierungsschätzfehler Quad (Boxplot)
- Gruppe B: Hit Rate vs. SNR (Linienplot)
- Gruppe B: Sondereffekte room_ir/mp3 (Balkendiagramm)
- Gruppe C: Real-World-Szenarien (Balkendiagramm)
- Gruppe D: Hit Rate vs. Query-Länge (Linienplot)
- OOD / Specificity pro System und Gruppe
- Gesamt-Übersichtstabelle
- Effizienz-Tabelle (LaTeX-Export)
- Dry-Run-Checks

In [ ]:
# ── RUN MODE CONFIG ──────────────────────────────────────────────────────────
# Switch between dry run and live run by editing src/run_config.py.
import sys
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve() / "src"))
from run_config import cfg
RUN_MODE = cfg.run_mode
print(f"RUN_MODE = {RUN_MODE!r}")

In [ ]:
import os, sys, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

PROJECT_ROOT = Path('..').resolve()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from run_config import cfg
from metrics import (
    compute_hit_rate, compute_precision, compute_specificity,
    compute_no_match_rate, compute_time_stats,
    compute_scale_estimation_error,
)
from utils import assert_disjoint, load_partitions

RUN_MODE = cfg.run_mode
SEED = 42
np.random.seed(SEED)

# ── Mode-dependent paths ──────────────────────────────────────────────────────
RESULTS_DIR    = PROJECT_ROOT / 'results' / cfg.results_subdir
PLOTS_DIR      = PROJECT_ROOT / 'results' / cfg.plots_subdir
MANIFEST_PATH  = PROJECT_ROOT / 'data' / 'queries' / cfg.manifest_name
PARTITIONS_DIR = PROJECT_ROOT / 'data' / 'partitions'

PLOTS_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({'figure.dpi': 150, 'font.size': 11})

SYSTEMS       = ['shazam', 'quad', 'neuralFP']
SYSTEM_COLORS = {'shazam': '#1f77b4', 'quad': '#ff7f0e', 'neuralFP': '#2ca02c'}
SYSTEM_LABELS = {'shazam': 'Shazam', 'quad': 'Quad', 'neuralFP': 'NeuralFP'}
print(f'RUN_MODE    = {RUN_MODE!r}')
print(f'RESULTS_DIR = {RESULTS_DIR}')
print(f'PLOTS_DIR   = {PLOTS_DIR}')
print('Setup abgeschlossen.')

## 1. Voraussetzungen prüfen & Daten laden

In [ ]:
required = {
    'shazam_raw.csv':         RESULTS_DIR / 'shazam_raw.csv',
    'quad_raw.csv':           RESULTS_DIR / 'quad_raw.csv',
    'neuralFP_raw.csv':       RESULTS_DIR / 'neuralFP_raw.csv',
    'manifest':               MANIFEST_PATH,
    'shazam_efficiency.json': RESULTS_DIR / 'shazam_efficiency.json',
    'quad_efficiency.json':   RESULTS_DIR / 'quad_efficiency.json',
}
all_ok = True
for label, p in required.items():
    status = 'OK' if p.exists() else 'FEHLT'
    print(f'  [{status}] {label}: {p}')
    if not p.exists():
        all_ok = False
if not all_ok:
    raise FileNotFoundError('Nicht alle Raw-Result-CSVs vorhanden. Pipelines 02/03/05 ausführen!')
print('Alle Voraussetzungen erfüllt. ✓')

In [ ]:
df_shazam = pd.read_csv(RESULTS_DIR / 'shazam_raw.csv')
df_quad   = pd.read_csv(RESULTS_DIR / 'quad_raw.csv')
df_neural = pd.read_csv(RESULTS_DIR / 'neuralFP_raw.csv')
df_all    = pd.concat([df_shazam, df_quad, df_neural], ignore_index=True)

# Ensure boolean dtype for is_ood (CSV may store as string 'True'/'False')
for _df in [df_shazam, df_quad, df_neural, df_all]:
    if _df['is_ood'].dtype == object:
        _df['is_ood'] = _df['is_ood'].map({'True': True, 'False': False})

manifest   = pd.read_csv(MANIFEST_PATH)
partitions = load_partitions(PARTITIONS_DIR)

print(f'Shazam:   {len(df_shazam)} Zeilen')
print(f'Quad:     {len(df_quad)} Zeilen')
print(f'NeuralFP: {len(df_neural)} Zeilen')
print(f'Gesamt:   {len(df_all)} Zeilen')
print(f'Bedingungen im Datensatz: {sorted(df_all["condition"].unique())}')
print()
print(df_all.head(3).to_string())

## 2. Gruppe A — Hit Rate vs. Skalierung

In [ ]:
def _condition_info(cond):
    """Return (dist_type, numeric_value) for a condition string."""
    if cond == 'A_original':
        return ('original', None)
    parts = cond.split('_')
    if len(parts) < 3:
        return ('unknown', 0)
    dtype = parts[1]  # tempo / pitch / speed
    vstr  = parts[2]
    if dtype == 'pitch':
        sign = -1 if vstr.startswith('m') else 1
        return (dtype, sign * int(vstr[1:]))
    return (dtype, int(vstr))

group_A = df_all[df_all['group'] == 'A'].copy()
group_A['dist_type'] = group_A['condition'].apply(lambda c: _condition_info(c)[0])
group_A['dist_val']  = group_A['condition'].apply(lambda c: _condition_info(c)[1])

# Add A_original to each sub-group at appropriate x position
orig_rows = group_A[group_A['dist_type'] == 'original'].copy()
sub_copies = []
for dt, xval in [('tempo', 100), ('speed', 100), ('pitch', 0)]:
    cp = orig_rows.copy()
    cp['dist_type'] = dt
    cp['dist_val'] = xval
    sub_copies.append(cp)
group_A_ext = pd.concat(
    [group_A[group_A['dist_type'] != 'original']] + sub_copies,
    ignore_index=True,
)

dist_meta = {
    'tempo': ('Tempo-Faktor (%)', 'Tempo'),
    'pitch': ('Pitch-Shift (Halbtöne)', 'Pitch'),
    'speed': ('Speed-Faktor (%)', 'Speed'),
}

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
fig.suptitle('Gruppe A — Hit Rate vs. Skalierung', fontsize=12)

for ax, (dt, (xlabel, title)) in zip(axes, dist_meta.items()):
    sub = group_A_ext[(group_A_ext['dist_type'] == dt) & (~group_A_ext['is_ood'])]
    for sys_name in SYSTEMS:
        sys_rows = sub[sub['system'] == sys_name]
        if sys_rows.empty:
            continue
        hr_rows = []
        for val, grp in sys_rows.groupby('dist_val'):
            tp = (grp['result_class'] == 'TP').sum()
            hr = tp / len(grp) if len(grp) > 0 else 0.0
            hr_rows.append({'dist_val': val, 'hit_rate': hr})
        hr_df = pd.DataFrame(hr_rows).sort_values('dist_val')
        ax.plot(
            hr_df['dist_val'], hr_df['hit_rate'],
            label=SYSTEM_LABELS[sys_name],
            color=SYSTEM_COLORS[sys_name],
            marker='o', linewidth=1.5,
        )
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    if ax is axes[0]:
        ax.set_ylabel('Hit Rate')
    ax.set_ylim(-0.05, 1.05)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
fig.savefig(PLOTS_DIR / 'groupA_scaling.pdf', bbox_inches='tight')
plt.show()
print(f'Gespeichert: {PLOTS_DIR / "groupA_scaling.pdf"}')

## 3. Gruppe A — Skalierungsschätzfehler Quad

In [ ]:
scale_err_df = compute_scale_estimation_error(df_all)
print(f'Quad Skalierungsschätzfehler: {len(scale_err_df)} Einträge')
print(scale_err_df.head())

if scale_err_df.empty:
    print('Keine Quad-Gruppe-A-Daten vorhanden — Plot übersprungen.')
else:
    # Nur Bedingungen mit sinnvollem Schätzwert (nicht-NaN detected_*)
    scale_err_df = scale_err_df.dropna(subset=['time_scale_err', 'freq_scale_err'])

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle('Gruppe A — Quad Skalierungsschätzfehler', fontsize=12)

    for ax, (col, label) in zip(
        axes,
        [('time_scale_err', 'Zeitachse |Δs_time|'),
         ('freq_scale_err', 'Frequenzachse |Δs_freq|')],
    ):
        cond_order = sorted(scale_err_df['condition'].unique())
        data_by_cond = [
            scale_err_df[scale_err_df['condition'] == c][col].values
            for c in cond_order
        ]
        bp = ax.boxplot(data_by_cond, labels=cond_order, patch_artist=True)
        for patch in bp['boxes']:
            patch.set_facecolor('#ff7f0e')
            patch.set_alpha(0.6)
        ax.set_title(label)
        ax.set_xlabel('Condition')
        ax.set_ylabel('Absoluter Schätzfehler')
        ax.tick_params(axis='x', rotation=45)
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    fig.savefig(PLOTS_DIR / 'scale_estimation.pdf', bbox_inches='tight')
    plt.show()
    print(f'Gespeichert: {PLOTS_DIR / "scale_estimation.pdf"}')

## 4. Gruppe B — Hit Rate vs. SNR

In [ ]:
# Filter: Gruppe B, SNR-Bedingungen, In-DB
group_B_snr = df_all[
    df_all['condition'].str.startswith('B_snr') &
    (df_all['group'] == 'B') &
    (~df_all['is_ood'])
].copy()

# Extract numeric SNR value from condition, e.g. 'B_snr_10' → 10, 'B_snr_m5' → -5
def _snr_value(cond):
    parts = cond.split('_')
    vstr = parts[-1]
    if vstr.startswith('m'):
        return -int(vstr[1:])
    return int(vstr)

group_B_snr['snr_db'] = group_B_snr['condition'].apply(_snr_value)

fig, ax = plt.subplots(figsize=(8, 4))
fig.suptitle('Gruppe B — Hit Rate vs. SNR', fontsize=12)

for sys_name in SYSTEMS:
    sys_rows = group_B_snr[group_B_snr['system'] == sys_name]
    if sys_rows.empty:
        continue
    hr_rows = []
    for snr_val, grp in sys_rows.groupby('snr_db'):
        tp = (grp['result_class'] == 'TP').sum()
        hr = tp / len(grp) if len(grp) > 0 else 0.0
        hr_rows.append({'snr_db': snr_val, 'hit_rate': hr})
    hr_df = pd.DataFrame(hr_rows).sort_values('snr_db')
    ax.plot(
        hr_df['snr_db'], hr_df['hit_rate'],
        label=SYSTEM_LABELS[sys_name],
        color=SYSTEM_COLORS[sys_name],
        marker='o', linewidth=1.5,
    )

ax.set_xlabel('SNR (dB)')
ax.set_ylabel('Hit Rate')
ax.set_ylim(-0.05, 1.05)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
fig.savefig(PLOTS_DIR / 'groupB_snr.pdf', bbox_inches='tight')
plt.show()
print(f'Gespeichert: {PLOTS_DIR / "groupB_snr.pdf"}')

In [ ]:
# SNR-Breakpoint: niedrigster SNR-Wert, bei dem Hit Rate noch >= 50%
# Nutzt group_B_snr und _snr_value aus der vorherigen Zelle.

if group_B_snr.empty:
    print('Keine Gruppe-B-SNR-Daten — Breakpoint-Tabelle übersprungen.')
else:
    bp_rows = []
    for sys_name in SYSTEMS:
        sys_rows = group_B_snr[group_B_snr['system'] == sys_name]
        hr_by_snr = []
        for snr_val, grp in sys_rows.groupby('snr_db'):
            tp = (grp['result_class'] == 'TP').sum()
            hr = tp / len(grp) if len(grp) > 0 else 0.0
            hr_by_snr.append({'snr_db': snr_val, 'hit_rate': hr})
        if not hr_by_snr:
            bp_rows.append({
                'System': SYSTEM_LABELS[sys_name],
                'Breakpoint SNR (dB)': float('nan'),
                'Hit Rate @ Breakpoint': float('nan'),
            })
            continue
        hr_df  = pd.DataFrame(hr_by_snr).sort_values('snr_db')
        above  = hr_df[hr_df['hit_rate'] >= 0.5]
        if above.empty:
            bp_snr = float('nan')
            bp_hr  = float('nan')
        else:
            bp_snr = float(above['snr_db'].min())
            bp_hr  = float(above.loc[above['snr_db'] == bp_snr, 'hit_rate'].iloc[0])
        bp_rows.append({
            'System':               SYSTEM_LABELS[sys_name],
            'Breakpoint SNR (dB)':  bp_snr,
            'Hit Rate @ Breakpoint': bp_hr,
        })

    bp_df = pd.DataFrame(bp_rows).set_index('System')
    print('SNR-Breakpoint-Tabelle (niedrigster SNR mit Hit Rate ≥ 50%):')
    print(bp_df.to_string(float_format=lambda x: f'{x:.3f}'))

    bp_path = RESULTS_DIR / 'snr_breakpoints.csv'
    bp_df.to_csv(bp_path)
    print(f'\nGespeichert: {bp_path} ✓')


## 5. Gruppe B — Sondereffekte

In [ ]:
fx_conditions = ['B_room_ir', 'B_mp3_128', 'B_mp3_64']
group_B_fx = df_all[
    df_all['condition'].isin(fx_conditions) &
    (~df_all['is_ood'])
].copy()

if group_B_fx.empty:
    print('Keine Gruppe-B-Sondereffekt-Daten vorhanden.')
else:
    fig, ax = plt.subplots(figsize=(8, 4))
    fig.suptitle('Gruppe B — Sondereffekte', fontsize=12)

    x = np.arange(len(fx_conditions))
    width = 0.25
    for i, sys_name in enumerate(SYSTEMS):
        sys_rows = group_B_fx[group_B_fx['system'] == sys_name]
        hrs = []
        for cond in fx_conditions:
            grp = sys_rows[sys_rows['condition'] == cond]
            tp  = (grp['result_class'] == 'TP').sum() if len(grp) > 0 else 0
            hrs.append(tp / len(grp) if len(grp) > 0 else 0.0)
        ax.bar(
            x + (i - 1) * width, hrs, width,
            label=SYSTEM_LABELS[sys_name],
            color=SYSTEM_COLORS[sys_name], alpha=0.8,
        )

    ax.set_xticks(x)
    ax.set_xticklabels(['Room IR', 'MP3 128kbps', 'MP3 64kbps'])
    ax.set_ylabel('Hit Rate')
    ax.set_ylim(0, 1.15)
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    fig.savefig(PLOTS_DIR / 'groupB_fx.pdf', bbox_inches='tight')
    plt.show()
    print(f'Gespeichert: {PLOTS_DIR / "groupB_fx.pdf"}')

## 6. Gruppe C — Real-World-Szenarien

In [ ]:
group_C = df_all[(df_all['group'] == 'C') & (~df_all['is_ood'])].copy()

if group_C.empty:
    print('Keine Gruppe-C-Daten vorhanden.')
else:
    c_conditions = sorted(group_C['condition'].unique())
    fig, ax = plt.subplots(figsize=(8, 4))
    fig.suptitle('Gruppe C — Real-World-Szenarien', fontsize=12)

    x = np.arange(len(c_conditions))
    width = 0.25
    for i, sys_name in enumerate(SYSTEMS):
        sys_rows = group_C[group_C['system'] == sys_name]
        hrs = []
        for cond in c_conditions:
            grp = sys_rows[sys_rows['condition'] == cond]
            tp  = (grp['result_class'] == 'TP').sum() if len(grp) > 0 else 0
            hrs.append(tp / len(grp) if len(grp) > 0 else 0.0)
        ax.bar(
            x + (i - 1) * width, hrs, width,
            label=SYSTEM_LABELS[sys_name],
            color=SYSTEM_COLORS[sys_name], alpha=0.8,
        )

    ax.set_xticks(x)
    ax.set_xticklabels([c.replace('C_', '') for c in c_conditions], rotation=20)
    ax.set_ylabel('Hit Rate')
    ax.set_ylim(0, 1.15)
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    fig.savefig(PLOTS_DIR / 'groupC_realworld.pdf', bbox_inches='tight')
    plt.show()
    print(f'Gespeichert: {PLOTS_DIR / "groupC_realworld.pdf"}')

## 7. Gruppe D — Hit Rate vs. Query-Länge

In [ ]:
group_D = df_all[(df_all['group'] == 'D') & (~df_all['is_ood'])].copy()

if group_D.empty:
    print('Keine Gruppe-D-Daten vorhanden.')
else:
    # Extract nominal length and noise level from condition
    # e.g. D_3s_clean → 3, clean; D_10s_snr10 → 10, snr10
    def _parse_D_condition(cond):
        # Format: D_clean_3s oder D_snr10_15s
        # parts[0]='D', parts[1]=noise, parts[-1]='Xs'
        parts = cond.split('_')
        if len(parts) < 3:
            return None, None
        dur_str = parts[-1].replace('s', '')   # ← war parts[1]
        noise   = '_'.join(parts[1:-1])        # ← war parts[2:]
        try:
            return int(dur_str), noise
        except ValueError:
            return None, None

    group_D['nominal_s'] = group_D['condition'].apply(lambda c: _parse_D_condition(c)[0])
    group_D['noise_lvl'] = group_D['condition'].apply(lambda c: _parse_D_condition(c)[1])
    group_D = group_D.dropna(subset=['nominal_s', 'noise_lvl'])

    noise_levels = sorted(group_D['noise_lvl'].unique())
    dur_values   = sorted(group_D['nominal_s'].unique())

    linestyles = {'clean': '-', 'snr10': '--'}

    fig, ax = plt.subplots(figsize=(9, 4))
    fig.suptitle('Gruppe D — Hit Rate vs. Query-Länge', fontsize=12)

    for sys_name in SYSTEMS:
        for noise in noise_levels:
            sub = group_D[
                (group_D['system'] == sys_name) &
                (group_D['noise_lvl'] == noise)
            ]
            if sub.empty:
                continue
            hr_rows = []
            for dur, grp in sub.groupby('nominal_s'):
                tp = (grp['result_class'] == 'TP').sum()
                hr_rows.append({'dur': dur, 'hit_rate': tp / len(grp)})
            hr_df = pd.DataFrame(hr_rows).sort_values('dur')
            ls = linestyles.get(noise, ':')
            ax.plot(
                hr_df['dur'], hr_df['hit_rate'],
                label=f'{SYSTEM_LABELS[sys_name]} ({noise})',
                color=SYSTEM_COLORS[sys_name],
                linestyle=ls, marker='o', linewidth=1.5,
            )

    ax.set_xlabel('Query-Länge (s)')
    ax.set_ylabel('Hit Rate')
    ax.set_ylim(-0.05, 1.05)
    ax.legend(fontsize=8, ncol=2)
    ax.grid(True, alpha=0.3)
    ax.set_xticks([3, 5, 10, 15])
    ax.set_xticklabels(['3', '5', '10', '15'])
    plt.tight_layout()
    fig.savefig(PLOTS_DIR / 'groupD_querylength.pdf', bbox_inches='tight')
    plt.show()
    print(f'Gespeichert: {PLOTS_DIR / "groupD_querylength.pdf"}')

## 8. OOD / Specificity

In [ ]:
print('=== Specificity + FPR pro System (gesamt) ===')
spec_overall = {}
for sys_name in SYSTEMS:
    spec = compute_specificity(df_all[df_all['system'] == sys_name])
    spec_overall[sys_name] = spec
    print(f'  {SYSTEM_LABELS[sys_name]}: Specificity={spec:.3f}  FPR={1 - spec:.3f}')

print()
print('=== Specificity + FPR pro System × Gruppe ===')
spec_by_group = {}
for sys_name in SYSTEMS:
    sys_df = df_all[df_all['system'] == sys_name]
    row = {}
    for grp in sorted(df_all['group'].unique()):
        spec = compute_specificity(sys_df[sys_df['group'] == grp])
        row[f'Spec_{grp}'] = spec
        row[f'FPR_{grp}']  = 1.0 - spec
    spec_by_group[sys_name] = row

spec_df = pd.DataFrame(spec_by_group).T
# Interleave columns: Spec_A, FPR_A, Spec_B, FPR_B, ...
groups_sorted = sorted(df_all['group'].unique())
col_order = [col for g in groups_sorted for col in (f'Spec_{g}', f'FPR_{g}')
             if f'Spec_{g}' in spec_df.columns]
spec_df = spec_df[[c for c in col_order if c in spec_df.columns]]
print()
print(spec_df.to_string(float_format=lambda x: f'{x:.3f}'))


In [ ]:
# Specificity bar chart per system and group
groups_present = sorted(df_all['group'].unique())
fig, ax = plt.subplots(figsize=(8, 4))
fig.suptitle('Specificity (OOD) — pro System & Gruppe', fontsize=12)

x = np.arange(len(groups_present))
width = 0.25
for i, sys_name in enumerate(SYSTEMS):
    sys_df = df_all[df_all['system'] == sys_name]
    specs = [
        compute_specificity(sys_df[sys_df['group'] == g])
        for g in groups_present
    ]
    ax.bar(
        x + (i - 1) * width, specs, width,
        label=SYSTEM_LABELS[sys_name],
        color=SYSTEM_COLORS[sys_name], alpha=0.8,
    )

ax.set_xticks(x)
ax.set_xticklabels([f'Gruppe {g}' for g in groups_present])
ax.set_ylabel('Specificity')
ax.set_ylim(0, 1.15)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
fig.savefig(PLOTS_DIR / 'specificity_ood.pdf', bbox_inches='tight')
plt.show()
print(f'Gespeichert: {PLOTS_DIR / "specificity_ood.pdf"}')

## 9. Gesamt-Übersichtstabelle

In [ ]:
# Hit Rate: system × gruppe (In-DB only)
rows = []
for sys_name in SYSTEMS:
    row = {'System': SYSTEM_LABELS[sys_name]}
    sys_df = df_all[df_all['system'] == sys_name]
    for grp in sorted(df_all['group'].unique()):
        hr = compute_hit_rate(sys_df[sys_df['group'] == grp])
        row[f'Gruppe {grp}'] = hr
    row['Gesamt'] = compute_hit_rate(sys_df)
    row['Precision'] = compute_precision(sys_df)
    row['Specificity'] = compute_specificity(sys_df)
    rows.append(row)

summary_df = pd.DataFrame(rows).set_index('System')
print('Gesamt-Übersichtstabelle:')
print(summary_df.to_string(float_format=lambda x: f'{x:.3f}'))

## 10. Effizienz-Tabelle

In [ ]:
eff_rows = []

# --- Shazam ---
shazam_eff_path = RESULTS_DIR / 'shazam_efficiency.json'
if shazam_eff_path.exists():
    with open(shazam_eff_path) as fh:
        se = json.load(fh)
    n_ref_sh      = se.get('n_ref_tracks', float('nan'))
    build_time_sh = se.get('build_time_s', float('nan'))
    db_bytes_sh   = se.get('index_size_mb', float('nan')) * 1024 * 1024
    shazam_ts     = compute_time_stats(df_shazam)
    eff_rows.append({
        'System':              'Shazam',
        'Build-Zeit (s)':      build_time_sh,
        'Build-Zeit/Song (s)': build_time_sh / n_ref_sh if not pd.isna(n_ref_sh) else float('nan'),
        'Index-Größe (MB)':    se.get('index_size_mb', float('nan')),
        'DB-Größe/Song (KB)':  db_bytes_sh / n_ref_sh / 1024 if not pd.isna(n_ref_sh) else float('nan'),
        'Median Query (ms)':   shazam_ts['median'],
        'P95 Query (ms)':      shazam_ts['p95'],
    })
else:
    eff_rows.append({'System': 'Shazam', 'Build-Zeit (s)': float('nan'),
                     'Build-Zeit/Song (s)': float('nan'), 'Index-Größe (MB)': float('nan'),
                     'DB-Größe/Song (KB)': float('nan'), 'Median Query (ms)': float('nan'),
                     'P95 Query (ms)': float('nan')})

# --- Quad ---
quad_eff_path = RESULTS_DIR / 'quad_efficiency.json'
if quad_eff_path.exists():
    with open(quad_eff_path) as fh:
        qe = json.load(fh)
    bs            = qe.get('build_stats', qe)
    n_ref_qd      = bs.get('processed', float('nan'))
    build_time_qd = bs.get('build_time_s', float('nan'))
    db_bytes_qd   = bs.get('db_memory_mb', float('nan')) * 1024 * 1024
    quad_ts       = compute_time_stats(df_quad)
    eff_rows.append({
        'System':              'Quad',
        'Build-Zeit (s)':      build_time_qd,
        'Build-Zeit/Song (s)': build_time_qd / n_ref_qd if not pd.isna(n_ref_qd) else float('nan'),
        'Index-Größe (MB)':    bs.get('db_memory_mb', float('nan')),
        'DB-Größe/Song (KB)':  db_bytes_qd / n_ref_qd / 1024 if not pd.isna(n_ref_qd) else float('nan'),
        'Median Query (ms)':   quad_ts['median'],
        'P95 Query (ms)':      quad_ts['p95'],
    })
else:
    eff_rows.append({'System': 'Quad', 'Build-Zeit (s)': float('nan'),
                     'Build-Zeit/Song (s)': float('nan'), 'Index-Größe (MB)': float('nan'),
                     'DB-Größe/Song (KB)': float('nan'), 'Median Query (ms)': float('nan'),
                     'P95 Query (ms)': float('nan')})

# --- NeuralFP ---
pfann_eff_path   = RESULTS_DIR / 'pfann_efficiency.json'
pfann_build_time = float('nan')
n_ref_pf         = float('nan')
if pfann_eff_path.exists():
    with open(pfann_eff_path) as fh:
        pe = json.load(fh)
    pfann_build_time = pe.get('build_time_s', float('nan'))
    n_ref_pf         = pe.get('n_ref_tracks', float('nan'))
# Fallback: count from partition JSON if efficiency file absent
if pd.isna(n_ref_pf) and 'live_ref' in partitions:
    n_ref_pf = float(len(partitions['live_ref']))
# DB size: sum all files in pfann_db/
pfann_db_dir   = RESULTS_DIR / 'pfann_db'
pfann_db_mb    = float('nan')
total_bytes_pf = float('nan')
if pfann_db_dir.exists():
    total_bytes_pf = float(sum(
        f.stat().st_size for f in pfann_db_dir.rglob('*') if f.is_file()
    ))
    pfann_db_mb = total_bytes_pf / (1024 ** 2)
neural_ts = compute_time_stats(df_neural)
eff_rows.append({
    'System':              'NeuralFP',
    'Build-Zeit (s)':      pfann_build_time,
    'Build-Zeit/Song (s)': pfann_build_time / n_ref_pf
                           if not (pd.isna(pfann_build_time) or pd.isna(n_ref_pf)) else float('nan'),
    'Index-Größe (MB)':    pfann_db_mb,
    'DB-Größe/Song (KB)':  total_bytes_pf / n_ref_pf / 1024
                           if not (pd.isna(total_bytes_pf) or pd.isna(n_ref_pf)) else float('nan'),
    'Median Query (ms)':   neural_ts['median'],
    'P95 Query (ms)':      neural_ts['p95'],
})

efficiency_df = pd.DataFrame(eff_rows).set_index('System')
print('Effizienz-Tabelle:')
print(efficiency_df.to_string(float_format=lambda x: f'{x:.2f}'))

# LaTeX export
latex_path = RESULTS_DIR / 'efficiency_table.tex'
efficiency_df.to_latex(
    latex_path,
    float_format='%.2f',
    na_rep='--',
)
print(f'\nLaTeX exportiert: {latex_path} ✓')
print(latex_path.read_text())


## 11. Compliance Checks

In [ ]:
if RUN_MODE == "dry":
    print('=== Dry-Run-Checks ===')
    errors = []

    # Check 1: Baseline Hit Rate A_original > 0.85 for Shazam & Quad
    hr_shazam_orig = compute_hit_rate(df_shazam, filter_col='condition', filter_val='A_original')
    hr_quad_orig   = compute_hit_rate(df_quad,   filter_col='condition', filter_val='A_original')
    print(f'[1] Shazam A_original Hit Rate : {hr_shazam_orig:.3f}', '✓' if hr_shazam_orig > 0.85 else 'FAIL')
    print(f'    Quad   A_original Hit Rate : {hr_quad_orig:.3f}',   '✓' if hr_quad_orig   > 0.85 else 'FAIL')
    if hr_shazam_orig <= 0.85: errors.append('Shazam A_original HR <= 0.85')
    if hr_quad_orig   <= 0.85: errors.append('Quad A_original HR <= 0.85')

    # Check 2: No data leakage
    try:
        parts = load_partitions(PARTITIONS_DIR)
        assert_disjoint(parts['dry_ref'], parts['dry_train'], parts['dry_ood'])
        print('[2] No data leakage ✓')
    except Exception as e:
        errors.append(f'Data leakage: {e}')
        print(f'[2] FAIL: {e}')

    # Check 3: OOD queries have ref_track_id = None
    manifest = pd.read_csv(MANIFEST_PATH)
    ood_null_ok = manifest[manifest['is_ood']]['ref_track_id'].isna().all()
    print(f'[3] OOD ref_track_id = None: {ood_null_ok}', '✓' if ood_null_ok else 'FAIL')
    if not ood_null_ok: errors.append('OOD has non-null ref_track_id')

    # Check 4: Speed conditions have duration != 10s
    speed_120 = manifest[manifest['condition'] == 'A_speed_120']
    speed_dur_ok = (speed_120['duration_sec'] < 9.5).all()
    print(f'[4] A_speed_120 duration < 9.5s: {speed_dur_ok}', '✓' if speed_dur_ok else 'FAIL')
    if not speed_dur_ok: errors.append('A_speed_120 duration not < 9.5s')

    # Check 5: pfann DB and matches exist
    tsv_ok = (RESULTS_DIR / 'neuralFP_matches.tsv').exists()
    print(f'[5] neuralFP_matches.tsv exists: {tsv_ok}', '✓' if tsv_ok else 'FAIL')
    if not tsv_ok: errors.append('neuralFP_matches.tsv missing')

    # Check 6: All four result classes present for all systems
    for sys_name in SYSTEMS:
        classes = df_all[df_all['system'] == sys_name]['result_class'].unique()
        has_tp = 'TP' in classes
        has_tn = 'TN' in classes
        print(f'[6] {sys_name}: TP={has_tp} TN={has_tn}', '✓' if (has_tp and has_tn) else 'check')

    if errors:
        print(f'\n{len(errors)} CHECK(S) FAILED:')
        for e in errors: print(f'  - {e}')
    else:
        print('\nAll dry-run checks passed. ✓')
else:
    print('=== Live-Run Compliance Checks ===')

    # Check 1: Baseline for Shazam & Quad (higher threshold for live run)
    hr_shazam_orig = compute_hit_rate(df_shazam, filter_col='condition', filter_val='A_original')
    hr_quad_orig   = compute_hit_rate(df_quad,   filter_col='condition', filter_val='A_original')
    print(f'[1] Shazam A_original Hit Rate : {hr_shazam_orig:.3f}', '✓' if hr_shazam_orig > 0.90 else 'check')
    print(f'    Quad   A_original Hit Rate : {hr_quad_orig:.3f}',   '✓' if hr_quad_orig   > 0.90 else 'check')

    # Check 2: No data leakage
    parts = load_partitions(PARTITIONS_DIR)
    assert_disjoint(parts['live_ref'], parts['live_train'], parts['live_ood'])
    print('[2] No data leakage (live) ✓')

    # Check 3: live_query ⊆ live_ref
    assert set(parts['live_query']) <= set(parts['live_ref']), 'live_query ⊄ live_ref!'
    print('[3] live_query ⊆ live_ref ✓')

    # Check 4: All four result classes present
    for sys_name in SYSTEMS:
        classes = df_all[df_all['system'] == sys_name]['result_class'].unique()
        print(f'[4] {sys_name}: result classes = {sorted(classes)}')

    print('\nLive-run checks done.')

## 12. Abschluss-Sanity

In [ ]:
print('=== df_all Shape & erste 5 Zeilen ===')
print(f'Shape: {df_all.shape}')
print(df_all.head(5).to_string())
print()
print(f'Plots gespeichert in {PLOTS_DIR}:')
for p in sorted(PLOTS_DIR.iterdir()):
    print(f'  {p.name}')